<a href="https://colab.research.google.com/github/ganeshmpsmg/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [20]:
import pandas as pd
import numpy as np

from google.colab import files

uploaded = files.upload()
file_name = next(iter(uploaded))

df = pd.read_csv(file_name)

print("Shape:", df.shape)
display(df.head())

Saving w05_ml_practice_dataset.csv to w05_ml_practice_dataset (2).csv
Shape: (100, 6)


,item_id,impressions,clicks,staleness_days,position,target
0,1,960,103,33,7,1
1,2,3872,688,1,6,0
2,3,3192,730,19,14,1
3,4,566,551,2,14,0
4,5,4526,743,44,6,1


## 1. Method choice and why

I chose Logistic Regression because it is a simple and interpretable method for a binary prediction task.

It gives a useful comparison against my Week-4 rule-based baseline without adding unnecessary complexity. The model coefficients also provide a directional view of which features influence the prediction.

The goal is to test whether the learned model provides useful improvement over the baseline, rather than adding complexity just for the sake of using a more advanced model.

## 2. Split design

I use an 80/20 stratified train-test split so both classes are represented in the test set.

The split is fixed with random_state=42 for reproducibility.

The baseline and model are evaluated on the same test observations and with the same metric so the comparison is fair.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [21]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)
pred = model.predict(X_test)

# Week-4 style baseline
baseline_score = (
    (X_test["impressions"] >= 1000).astype(int) * 2
    + (X_test["staleness_days"] >= 14).astype(int) * 2
    + (X_test["position"] >= 8).astype(int)
)

baseline_pred = (baseline_score >= 3).astype(int)

baseline_accuracy = accuracy_score(y_test, baseline_pred)
model_accuracy = accuracy_score(y_test, pred)

comparison = pd.DataFrame({
    "Method": [
        "Week-4 Baseline",
        "Logistic Regression"
    ],
    "Accuracy": [
        baseline_accuracy,
        model_accuracy
    ]
})

display(comparison)

,Method,Accuracy
0,Week-4 Baseline,0.7
1,Logistic Regression,0.8


In [22]:
from sklearn.model_selection import train_test_split

features = ["impressions", "clicks", "staleness_days", "position"]

X = df[features]
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training:", X_train.shape)
print("Testing:", X_test.shape)

Training: (80, 4)
Testing: (20, 4)


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [23]:
errors = pd.DataFrame({
    "actual": y_test.values,
    "predicted": pred
})

errors["wrong"] = errors["actual"] != errors["predicted"]

print("Total errors:", errors["wrong"].sum())

display(errors[errors["wrong"]].head(10))

Total errors: 4


,actual,predicted,wrong
5,0,1,True
9,0,1,True
10,0,1,True
12,0,1,True


In [24]:
coef = model.named_steps["model"].coef_[0]

importance = pd.DataFrame({
    "feature": features,
    "coefficient": coef
})

importance["absolute_coefficient"] = importance["coefficient"].abs()

display(
    importance.sort_values(
        "absolute_coefficient",
        ascending=False
    )
)

,feature,coefficient,absolute_coefficient
2,staleness_days,0.640502,0.640502
0,impressions,0.432939,0.432939
1,clicks,0.113056,0.113056
3,position,0.104730,0.104730


## 4. Errors and interpretation

I inspected the incorrect predictions rather than relying only on the overall accuracy.

The observed errors show that the available features do not perfectly separate the two target outcomes. The Logistic Regression coefficients provide directional information about which features have the strongest relationship with the prediction.

The model is therefore treated as decision-support rather than a guaranteed prediction.

## Self-check

Before you submit, confirm each line honestly:

- [yes] Every section above is filled — markdown thinking AND the code that backs it
- [yes ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ yes] No client names, URLs, or private queries anywhere
- [ yes] My claims use careful words: observed, measured, directional, decision-support
- [yes ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.